In [1]:
import time 
import numpy as np 
import pandas as pd 
from IPython.display import display 
from pgmpy.causal_discovery import GES 
from pgmpy.structure_score import get_scoring_method 
from pgmpy.base import PDAG 
from pgmpy.example_models import load_model

c:\Users\Anusa\anaconda3\envs\pgmpy\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_bnlearn_dataset(dataset_name, n_samples=5000, seed=42):
    """ Loads and simulates a bnlearn dataset. """ 
    model = load_model( f"bnlearn/{dataset_name}" ) 
    data = model.simulate(n_samples=n_samples, seed=seed) 
    return data.astype("category")

In [3]:
def compute_graph_score(model, scoring_method, data): 
    """ Computes total graph score. """ 
    score = get_scoring_method(scoring_method, data) 
    score_fn = score.local_score 
    if isinstance(model, PDAG): 
        model = model.to_dag() 
    total_score = 0 
    for node in model.nodes(): 
        parents = tuple(model.predecessors(node)) 
        total_score += score_fn(node, parents) 
    return total_score

In [4]:
def benchmark_grid(data, scoring_method, dataset_name): 
    n_jobs_values = [1, 2, 4, 8, 16] 
    max_parent_values = [1, 2, 4, 8, 16] 
    results = [] 
    print("\n" + "=" * 80) 
    print(f"Running benchmark for: {dataset_name}") 
    print("=" * 80) 
    for n_jobs in n_jobs_values: 
        for max_parents in max_parent_values: 
            print( f"Running " f"n_jobs={n_jobs}, " f"max_parents={max_parents}" ) 
            start = time.perf_counter() 
            est = GES(variant="fges", scoring_method=scoring_method, n_jobs=n_jobs, max_parents=max_parents) 
            est.fit(data) 
            runtime = (time.perf_counter() - start) 
            graph_score = compute_graph_score(est.causal_graph_, scoring_method, data) 
            results.append({"dataset": dataset_name, "n_jobs": n_jobs, "max_parents": max_parents, "runtime_sec": round(runtime, 4), "graph_score": round(graph_score, 4), "num_edges": len( est.causal_graph_.edges()),}) 
        results_df = pd.DataFrame(results)
         
    return results_df

In [5]:
datasets = [ "sachs", "child", "alarm", "asia", "cancer"] 
all_results = [] 
for dataset_name in datasets: 
    data = load_bnlearn_dataset(dataset_name, n_samples=5000, seed=42) 
    results_df = benchmark_grid(data=data, scoring_method="bic-d", dataset_name=dataset_name.capitalize()) 
    all_results.append(results_df) 
    print( f"\nBenchmark Results: " f"{dataset_name.upper()}" ) 
    display(results_df)

D:\pgmpy\pgmpy\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (
Generating for node: Akt: 100%|██████████| 11/11 [00:00<00:00, 14.08it/s]



Running benchmark for: Sachs
Running n_jobs=1, max_parents=1
Running n_jobs=1, max_parents=2
Running n_jobs=1, max_parents=4
Running n_jobs=1, max_parents=8
Running n_jobs=1, max_parents=16
Running n_jobs=2, max_parents=1
Running n_jobs=2, max_parents=2
Running n_jobs=2, max_parents=4
Running n_jobs=2, max_parents=8
Running n_jobs=2, max_parents=16
Running n_jobs=4, max_parents=1
Running n_jobs=4, max_parents=2
Running n_jobs=4, max_parents=4
Running n_jobs=4, max_parents=8
Running n_jobs=4, max_parents=16
Running n_jobs=8, max_parents=1
Running n_jobs=8, max_parents=2
Running n_jobs=8, max_parents=4
Running n_jobs=8, max_parents=8
Running n_jobs=8, max_parents=16
Running n_jobs=16, max_parents=1
Running n_jobs=16, max_parents=2
Running n_jobs=16, max_parents=4
Running n_jobs=16, max_parents=8
Running n_jobs=16, max_parents=16

Benchmark Results: SACHS


,dataset,n_jobs,max_parents,runtime_sec,graph_score,num_edges
0,Sachs,1,1,0.2185,-39427.0602,14
1,Sachs,1,2,0.3582,-37834.9325,26
2,Sachs,1,4,0.4728,-37611.7686,28
3,Sachs,1,8,0.4690,-37611.7686,28
4,Sachs,1,16,0.4707,-37611.7686,28
5,Sachs,2,1,0.4276,-39427.0602,14
6,Sachs,2,2,1.2155,-37834.9325,26
7,Sachs,2,4,1.6817,-37611.7686,28
8,Sachs,2,8,1.0265,-37611.7686,28
9,Sachs,2,16,1.3286,-37611.7686,28


Generating for node: GruntingReport: 100%|██████████| 20/20 [00:00<00:00, 88.05it/s]



Running benchmark for: Child
Running n_jobs=1, max_parents=1
Running n_jobs=1, max_parents=2
Running n_jobs=1, max_parents=4
Running n_jobs=1, max_parents=8
Running n_jobs=1, max_parents=16
Running n_jobs=2, max_parents=1
Running n_jobs=2, max_parents=2
Running n_jobs=2, max_parents=4
Running n_jobs=2, max_parents=8
Running n_jobs=2, max_parents=16
Running n_jobs=4, max_parents=1
Running n_jobs=4, max_parents=2
Running n_jobs=4, max_parents=4
Running n_jobs=4, max_parents=8
Running n_jobs=4, max_parents=16
Running n_jobs=8, max_parents=1
Running n_jobs=8, max_parents=2
Running n_jobs=8, max_parents=4
Running n_jobs=8, max_parents=8
Running n_jobs=8, max_parents=16
Running n_jobs=16, max_parents=1
Running n_jobs=16, max_parents=2
Running n_jobs=16, max_parents=4
Running n_jobs=16, max_parents=8
Running n_jobs=16, max_parents=16

Benchmark Results: CHILD


...


Running benchmark for: Asia
Running n_jobs=1, max_parents=1
Running n_jobs=1, max_parents=2
Running n_jobs=1, max_parents=4
Running n_jobs=1, max_parents=8
Running n_jobs=1, max_parents=16
Running n_jobs=2, max_parents=1
Running n_jobs=2, max_parents=2
Running n_jobs=2, max_parents=4
Running n_jobs=2, max_parents=8
Running n_jobs=2, max_parents=16
Running n_jobs=4, max_parents=1
Running n_jobs=4, max_parents=2
Running n_jobs=4, max_parents=4
Running n_jobs=4, max_parents=8
Running n_jobs=4, max_parents=16
Running n_jobs=8, max_parents=1
Running n_jobs=8, max_parents=2
Running n_jobs=8, max_parents=4
Running n_jobs=8, max_parents=8
Running n_jobs=8, max_parents=16
Running n_jobs=16, max_parents=1
Running n_jobs=16, max_parents=2
Running n_jobs=16, max_parents=4
Running n_jobs=16, max_parents=8
Running n_jobs=16, max_parents=16

Benchmark Results: ASIA


,dataset,n_jobs,max_parents,runtime_sec,graph_score,num_edges
0,Asia,1,1,0.1161,-11280.2552,7
1,Asia,1,2,0.1108,-11189.0062,9
2,Asia,1,4,0.1002,-11189.0062,9
3,Asia,1,8,0.1026,-11189.0062,9
4,Asia,1,16,0.1593,-11189.0062,9
5,Asia,2,1,0.4278,-11280.2552,7
6,Asia,2,2,0.5615,-11189.0062,9
7,Asia,2,4,0.5996,-11189.0062,9
8,Asia,2,8,0.5712,-11189.0062,9
9,Asia,2,16,0.6243,-11189.0062,9


Generating for node: Dyspnoea: 100%|██████████| 5/5 [00:00<00:00, 197.62it/s]



Running benchmark for: Cancer
Running n_jobs=1, max_parents=1
Running n_jobs=1, max_parents=2
Running n_jobs=1, max_parents=4
Running n_jobs=1, max_parents=8
Running n_jobs=1, max_parents=16
Running n_jobs=2, max_parents=1
Running n_jobs=2, max_parents=2
Running n_jobs=2, max_parents=4
Running n_jobs=2, max_parents=8
Running n_jobs=2, max_parents=16
Running n_jobs=4, max_parents=1
Running n_jobs=4, max_parents=2
Running n_jobs=4, max_parents=4
Running n_jobs=4, max_parents=8
Running n_jobs=4, max_parents=16
Running n_jobs=8, max_parents=1
Running n_jobs=8, max_parents=2
Running n_jobs=8, max_parents=4
Running n_jobs=8, max_parents=8
Running n_jobs=8, max_parents=16
Running n_jobs=16, max_parents=1
Running n_jobs=16, max_parents=2
Running n_jobs=16, max_parents=4
Running n_jobs=16, max_parents=8
Running n_jobs=16, max_parents=16

Benchmark Results: CANCER


,dataset,n_jobs,max_parents,runtime_sec,graph_score,num_edges
0,Cancer,1,1,0.0386,-10390.329,6
1,Cancer,1,2,0.0478,-10390.329,6
2,Cancer,1,4,0.0408,-10390.329,6
3,Cancer,1,8,0.0442,-10390.329,6
4,Cancer,1,16,0.0719,-10390.329,6
5,Cancer,2,1,0.1672,-10390.329,6
6,Cancer,2,2,0.2482,-10390.329,6
7,Cancer,2,4,0.1769,-10390.329,6
8,Cancer,2,8,0.1594,-10390.329,6
9,Cancer,2,16,0.1799,-10390.329,6


In [6]:
def benchmark_plain_ges(
    data,
    scoring_method,
    dataset_name,
):
    results = []
    print("\n" + "=" * 80)
    print(
        f"Running baseline GES for: "
        f"{dataset_name}"
    )
    print("=" * 80)
    start = time.perf_counter()
    est = GES(scoring_method=scoring_method)
    est.fit(data)
    runtime = time.perf_counter() - start
    graph_score = compute_graph_score(est.causal_graph_, scoring_method, data)
    results.append(
        {
            "dataset": dataset_name,
            "runtime_sec": round(runtime, 4),
            "graph_score": round(graph_score, 4),
            "num_edges": len(
                est.causal_graph_.edges()
            ),
        }
    )

    results_df = pd.DataFrame(results)

    return results_df


In [7]:
datasets = [ "sachs", "child", "alarm", "asia", "cancer"] 
all_results = [] 
for dataset_name in datasets: 
    data = load_bnlearn_dataset(dataset_name, n_samples=5000, seed=42) 
    results_df = benchmark_plain_ges(data=data, scoring_method="bic-d", dataset_name=dataset_name.capitalize()) 
    all_results.append(results_df) 
    print( f"\nBenchmark Results: " f"{dataset_name.upper()}" ) 
    display(results_df)

Generating for node: Akt: 100%|██████████| 11/11 [00:00<00:00, 112.96it/s]



Running baseline GES for: Sachs

Benchmark Results: SACHS


,dataset,runtime_sec,graph_score,num_edges
0,Sachs,0.7662,-36278.1836,34


Generating for node: GruntingReport: 100%|██████████| 20/20 [00:00<00:00, 92.13it/s] 



Running baseline GES for: Child

Benchmark Results: CHILD


...


Running baseline GES for: Asia

Benchmark Results: ASIA


,dataset,runtime_sec,graph_score,num_edges
0,Asia,0.2197,-11189.0062,9


Generating for node: Dyspnoea: 100%|██████████| 5/5 [00:00<00:00, 170.31it/s]


Running baseline GES for: Cancer

Benchmark Results: CANCER


,dataset,runtime_sec,graph_score,num_edges
0,Cancer,0.045,-10390.329,6
